# Sentiment Comparison: FinBERT vs RoBERTa vs FinBERT-Tone

This notebook compares three transformer models on a dataset of financial market news:

- **FinBERT** ([`ProsusAI/finbert`](https://huggingface.co/ProsusAI/finbert)) — BERT fine-tuned on the Financial PhraseBank.
- **RoBERTa** ([`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)) — a general-purpose (non-financial) sentiment RoBERTa, used here as a baseline contrast to the finance-tuned models.
- **FinBERT-Tone** ([`yiyanghkust/finbert-tone`](https://huggingface.co/yiyanghkust/finbert-tone)) — BERT fine-tuned on analyst reports for financial tone classification.

All three models predict `positive` / `negative` / `neutral`, so their outputs are directly comparable.

**What this notebook does:**
1. Downloads your dataset from Google Drive.
2. Loads it and auto-detects the text column (override if needed).
3. Runs all three models over every row, in batches.
4. Saves a CSV with the original data plus each model's predicted label and per-class probabilities.
5. Produces a short comparison: label distributions, pairwise agreement, and disagreement examples.

> Run this in an environment with internet access (e.g. Google Colab) — a GPU is recommended but not required.


## 1. Install dependencies

In [ ]:
%pip install -q gdown transformers torch pandas tqdm scikit-learn matplotlib


## 2. Configuration

Edit the values in this cell to match your dataset.

In [ ]:
import os

# --- Google Drive source -----------------------------------------------
GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1q_mca4_Uk3oHkHHMnCtyBL8QWZirR05c"
DATA_DIR = "data"  # local folder the Drive folder will be downloaded into

# --- Dataset options -----------------------------------------------------
# Path to a specific file inside DATA_DIR. Leave as None to auto-pick the
# first CSV/XLSX found in DATA_DIR.
DATA_FILE = None

# Name of the column containing the news text to classify.
# Leave as "auto" to auto-detect from a list of common column names.
TEXT_COLUMN = "auto"

# Optional: name of an existing ground-truth sentiment column, if your
# dataset has one (e.g. from a labeled Kaggle dataset). Leave as "auto" to
# auto-detect, or set to None if there is no ground truth.
LABEL_COLUMN = "auto"

# --- Output ---------------------------------------------------------------
OUTPUT_CSV = "sentiment_comparison_results.csv"

# --- Inference options ------------------------------------------------
BATCH_SIZE = 32
MAX_LENGTH = 512


## 3. Download the dataset from Google Drive

In [ ]:
import gdown

os.makedirs(DATA_DIR, exist_ok=True)
gdown.download_folder(
    GDRIVE_FOLDER_URL,
    output=DATA_DIR,
    quiet=False,
    use_cookies=False,
)


## 4. Load the dataset

In [ ]:
import glob
import pandas as pd

def find_data_file(data_dir, explicit_path=None):
    if explicit_path:
        return explicit_path
    candidates = sorted(
        glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)
        + glob.glob(os.path.join(data_dir, "**", "*.xlsx"), recursive=True)
        + glob.glob(os.path.join(data_dir, "**", "*.xls"), recursive=True)
    )
    if not candidates:
        raise FileNotFoundError(
            f"No CSV/XLSX files found under '{data_dir}'. "
            "Set DATA_FILE explicitly to the path of your dataset."
        )
    return candidates[0]

data_path = find_data_file(DATA_DIR, DATA_FILE)
print(f"Loading: {data_path}")

if data_path.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(data_path)
else:
    # A few common finance-news CSVs (e.g. Financial PhraseBank exports) are
    # semicolon-delimited with no header and latin-1 encoded. Try the
    # standard format first and fall back if it looks malformed.
    try:
        df = pd.read_csv(data_path)
        if df.shape[1] == 1:
            raise ValueError("Only one column parsed, retrying with ';' delimiter")
    except (ValueError, UnicodeDecodeError):
        df = pd.read_csv(data_path, sep=";", header=None, encoding="latin-1",
                          names=["sentiment", "text"])

print(df.shape)
df.head()


## 5. Detect the text (and optional label) column

In [ ]:
TEXT_CANDIDATES = [
    "text", "Text", "news", "News", "headline", "Headline", "title", "Title",
    "sentence", "Sentence", "content", "Content", "article", "Article",
    "News Headline", "body", "Body",
]
LABEL_CANDIDATES = [
    "sentiment", "Sentiment", "label", "Label", "target", "Target", "class", "Class",
]

def resolve_column(df, configured, candidates, required=True):
    if configured not in (None, "auto"):
        if configured not in df.columns:
            raise KeyError(f"Configured column '{configured}' not found in {list(df.columns)}")
        return configured
    if configured is None:
        return None
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"Could not auto-detect a text column among {list(df.columns)}. "
            "Set TEXT_COLUMN explicitly in the config cell."
        )
    return None

text_col = resolve_column(df, TEXT_COLUMN, TEXT_CANDIDATES, required=True)
label_col = resolve_column(df, LABEL_COLUMN, LABEL_CANDIDATES, required=False)

print(f"Text column:  {text_col!r}")
print(f"Label column: {label_col!r}")

df = df.dropna(subset=[text_col]).reset_index(drop=True)
df[text_col] = df[text_col].astype(str)
len(df)


## 6. Load the models

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'cuda' if device == 0 else 'cpu'}")

MODELS = {
    "finbert": "ProsusAI/finbert",
    "roberta": "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "finbert_tone": "yiyanghkust/finbert-tone",
}

pipelines = {}
for key, model_name in MODELS.items():
    print(f"Loading {key} ({model_name})...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipelines[key] = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=device,
        top_k=None,          # return scores for every class
        truncation=True,
        max_length=MAX_LENGTH,
    )
print("All models loaded.")


## 6b. Quick demo on toy examples

A tiny sanity check before committing to the full dataset: six hand-written headlines with obvious sentiment, classified by all three models side by side.

**You can run this without downloading any data** — only cells 1 (install), 2 (config), and 6 (load models) need to run first; the Drive download (sections 3–5) can be skipped.

Things to notice in the output:
- The two finance-tuned models usually call the rate-decision headline `neutral`, while the general-purpose RoBERTa often reads plain factual finance statements as negative or positive.
- The `_score` column is the winning class's probability — how confident each model is.

In [ ]:
toy_headlines = [
    "Company X shares surge 12% after record quarterly earnings beat expectations",
    "Regulators fine the bank $2 billion over money-laundering failures",
    "The central bank kept interest rates unchanged, as widely expected",
    "Tech giant announces layoffs of 10,000 employees amid slowing demand",
    "Oil prices edge higher on supply concerns; analysts remain cautious",
    "Startup files for bankruptcy after failing to secure new funding",
]

demo_rows = []
for text in toy_headlines:
    row = {"text": text}
    for key, pipe in pipelines.items():
        # pipe returns a list with one entry per input; that entry is a
        # list of {'label', 'score'} dicts, one per class
        scores = {d["label"].lower(): d["score"] for d in pipe(text)[0]}
        top = max(scores, key=scores.get)
        row[f"{key}_label"] = top
        row[f"{key}_score"] = round(scores[top], 3)
    demo_rows.append(row)

import pandas as pd
demo_df = pd.DataFrame(demo_rows)
demo_df


## 7. Run inference

Each model returns a probability for every class; we keep the full distribution plus the top predicted label.

In [ ]:
from tqdm.auto import tqdm

def run_pipeline(pipe, texts, batch_size=BATCH_SIZE):
    """Returns a list of {label: score, ...} dicts, one per text."""
    results = []
    for out in tqdm(pipe(texts, batch_size=batch_size), total=len(texts)):
        # out is a list of {'label':..., 'score':...} for each class
        results.append({d["label"].lower(): d["score"] for d in out})
    return results

texts = df[text_col].tolist()

raw_results = {}
for key, pipe in pipelines.items():
    print(f"Running {key}...")
    raw_results[key] = run_pipeline(pipe, texts)


## 8. Build result columns and merge with the original data

In [ ]:
def attach_results(df, prefix, raw):
    all_labels = sorted({lbl for row in raw for lbl in row.keys()})
    out = df.copy()
    for lbl in all_labels:
        out[f"{prefix}_{lbl}"] = [row.get(lbl, 0.0) for row in raw]
    top_labels, top_scores = [], []
    for row in raw:
        top_lbl = max(row, key=row.get)
        top_labels.append(top_lbl)
        top_scores.append(row[top_lbl])
    out[f"{prefix}_label"] = top_labels
    out[f"{prefix}_score"] = top_scores
    return out

result_df = df.copy()
for key, raw in raw_results.items():
    result_df = attach_results(result_df, key, raw)

result_df.head()


## 9. Save to CSV

In [ ]:
result_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(result_df)} rows with predictions to '{OUTPUT_CSV}'")


## 10. Compare the models

### 10.1 Label distribution per model

In [ ]:
import matplotlib.pyplot as plt

label_cols = {key: f"{key}_label" for key in pipelines}

dist = pd.DataFrame({
    key: result_df[col].value_counts(normalize=True)
    for key, col in label_cols.items()
}).fillna(0).sort_index()

dist.plot(kind="bar", figsize=(8, 5), title="Predicted label distribution by model")
plt.ylabel("Share of articles")
plt.xlabel("Label")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

dist


### 10.2 Pairwise agreement between models

In [ ]:
from itertools import combinations
from sklearn.metrics import cohen_kappa_score

model_keys = list(pipelines.keys())
agreement_rows = []
for a, b in combinations(model_keys, 2):
    la, lb = result_df[label_cols[a]], result_df[label_cols[b]]
    agreement = (la == lb).mean()
    kappa = cohen_kappa_score(la, lb)
    agreement_rows.append({"model_a": a, "model_b": b, "agreement": agreement, "cohen_kappa": kappa})

agreement_df = pd.DataFrame(agreement_rows)
agreement_df


### 10.3 Accuracy vs. ground truth (if a label column was found)

In [ ]:
if label_col is not None:
    normalized_truth = result_df[label_col].astype(str).str.lower()
    acc_rows = []
    for key, col in label_cols.items():
        acc = (result_df[col] == normalized_truth).mean()
        acc_rows.append({"model": key, "accuracy": acc})
    display(pd.DataFrame(acc_rows))
else:
    print("No ground-truth label column detected/configured — skipping accuracy comparison.")


### 10.4 Examples where the three models disagree

In [ ]:
disagreements = result_df[
    (result_df[label_cols["finbert"]] != result_df[label_cols["roberta"]])
    | (result_df[label_cols["finbert"]] != result_df[label_cols["finbert_tone"]])
]

cols_to_show = [text_col] + list(label_cols.values())
disagreements[cols_to_show].head(20)


## Next steps

- Adjust `TEXT_COLUMN` / `LABEL_COLUMN` in the config cell if auto-detection picked the wrong field.
- `sentiment_comparison_results.csv` contains the original data plus, for each model, a `<model>_label`, `<model>_score`, and one `<model>_<class>` probability column per class — ready for further analysis (e.g. joining with price data for an event study).
- Swap in any other Hugging Face model by adding an entry to the `MODELS` dict in section 6.
